# 23장 · Retriever의 검색 입력과 반환 형식

Retriever는 질문을 받아 관련 문서를 돌려주는 구성 요소입니다. 몇 건을 반환할지와 어떤 출처를 허용할지도 입력 조건이 됩니다.

벡터 저장소에서 받은 ID·본문·메타데이터의 중첩 목록을 문서별 레코드로 변환합니다. 질문이 여러 개일 때도 각 후보의 본문과 원문 쪽 번호가 함께 전달돼야 합니다.

실행을 확인한 커널은 Python 3.12.15입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 과제 셀에 직접 코드를 작성한 뒤 바로 아래에서 풀이를 펼쳐 보세요.


## 검색 결과의 중첩 목록

Chroma의 질의 응답은 여러 질문을 한 번에 받을 수 있게 질문별로 다시 묶여 있습니다. 아래 자료는 실제 서버를 호출하지 않고 그 모양만 담아 둔 한 질문의 응답 예시입니다.


### 셀 01 · 중첩 검색 결과의 인덱싱

첫 질문의 검색 결과에서 1위 후보의 문서 ID를 읽습니다.


In [1]:
raw = {"ids": [["library-1"]], "documents": [["도서 대출 기간은 14일입니다."]],
       "metadatas": [[{"source": "가상 도서관 이용 안내", "page": 1}]]}
print(raw["ids"][0][0])

library-1


library-1이 나옵니다. [0]을 한 번만 쓰면 문자열이 아니라 첫 질문의 ID 목록이 남습니다.

질문 두 개를 한 번에 검색하면 결과의 바깥 목록에도 항목이 두 개 생깁니다. 바깥쪽 인덱스는 어느 질문인지, 안쪽 인덱스는 그 질문에서 몇 번째 후보인지 고릅니다. 첫 질문의 첫 답에 쪽 번호를 붙일 때는 ids와 metadatas 양쪽에서 같은 두 위치를 읽어야 합니다.


### 질의 번호와 결과 순위 읽기

<p>raw['ids'][0][0]에서 첫 번째 0은 첫 질의, 두 번째 0은 첫 후보를 뜻합니다. 본문과 메타데이터에서도 같은 두 위치를 골라야 한 문서의 정보가 됩니다.</p>


### 과제 02 · 본문과 출처 필드 구성

세 열의 같은 위치를 사용해 id, text, source, page가 있는 사전을 만드세요.


In [2]:
# ID와 본문, 메타데이터를 한 건으로 합치세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>meta = raw[&quot;metadatas&quot;][0][0]
row = {&quot;id&quot;: raw[&quot;ids&quot;][0][0], &quot;text&quot;: raw[&quot;documents&quot;][0][0],
       &quot;source&quot;: meta[&quot;source&quot;], &quot;page&quot;: meta[&quot;page&quot;]}
print(row)</code></pre>
<p>실행 결과</p><pre><code>{&#x27;id&#x27;: &#x27;library-1&#x27;, &#x27;text&#x27;: &#x27;도서 대출 기간은 14일입니다.&#x27;, &#x27;source&#x27;: &#x27;가상 도서관 이용 안내&#x27;, &#x27;page&#x27;: 1}</code></pre>
<p>library-1과 1쪽이 같은 사전에 들어갑니다. 여러 결과를 합칠 때는 각 열의 길이가 같은지도 확인합니다. 출처가 누락된 행을 앞 행의 출처로 채우면 안 됩니다.</p>
</details>


### 준비된 결과로 반환 모양 확인하기

<p>raw는 첫 검색 결과를 미리 담아 둔 딕셔너리입니다. 이 값으로 ID·본문·출처·쪽을 연결하는 코드를 먼저 확인한 뒤, 다음 셀에서 retrieve가 실제 후보 순서를 정하는 과정을 살펴봅니다.</p>


## retrieve()의 k와 메타데이터 필터

retrieve(question, k)는 문서 목록을 돌려줍니다. 여기서는 세 키워드의 거리로 계산하는 로컬 검색을 실행합니다. 앞의 Chroma 모양 예시를 실제 조회했다는 뜻은 아닙니다.


### Retriever 함수의 역할

<p>검색을 화면이나 답변 생성 코드 안에 섞으면 저장소를 바꾸거나 실패 원인을 찾기 어려워집니다. retrieve(question,k)는 질문과 최대 후보 수를 받아 문서 목록을 반환하는 경계로 둡니다.</p>

<p>반환 문서는 id·text·source·page 네 필드를 유지합니다. 이후 LCEL과 LangGraph에서 같은 계약을 사용하면 검색 결과를 처리하는 부분을 재사용할 수 있습니다. 계약이 같아도 실제 검색 알고리즘과 품질이 같다는 뜻은 아닙니다.</p>

<table><caption>공통 계약</caption><thead><tr><th scope="col">입력·출력</th><th scope="col">뜻</th></tr>
</thead><tbody><tr><td>question: str</td><td>검색할 질문</td></tr>
<tr><td>k: int</td><td>반환 후보 수의 상한</td></tr>
<tr><td>list[dict]</td><td>검색된 문서들, 없으면 빈 목록</td></tr>
<tr><td>id/text/source/page</td><td>식별자·본문·출처·원문 위치</td></tr>
</tbody></table>

<p>공백을 제거한 질문이 비어 있으면 입력 오류입니다. k는 1 이상의 정수이며 bool은 제외합니다. Python의 True가 정수처럼 취급될 수 있어 따로 확인합니다. 검색 결과가 없는 정상 상황은 []이고, 출처가 잘못되거나 저장소가 응답하지 않는 상황은 예외입니다.</p>


### 셀 03 · retrieve()의 검색 개수

자료의 원문 ID를 유지합니다. k는 최대 후보 수이고, 질문은 공백을 제외한 문자열이어야 합니다.

휴관 질문에서는 결과 한 건이 있지만 주차 요금에는 대응하는 단어가 없어 빈 목록이 돌아옵니다. 질문을 바꾸면 첫 항목을 읽기 전 목록에 문서가 있는지 확인하세요.


In [3]:
documents = [
  {"id":"library-1","text":"도서 대출 기간은 14일입니다.","source":"가상 도서관 이용 안내","page":1},
  {"id":"library-2","text":"도서관은 월요일에 휴관합니다.","source":"가상 도서관 이용 안내","page":2},
  {"id":"library-3","text":"무인 반납함은 24시간 이용할 수 있습니다.","source":"가상 도서관 이용 안내","page":3}
]

terms = ("대출", "휴관", "반납")

def teaching_vector(text):
    return [float(term in text) for term in terms]

def validate_request(question, k=2):
    if not isinstance(question, str) or not question.strip():
        raise ValueError("질문은 비어 있지 않은 문자열이어야 합니다.")
    if isinstance(k, bool) or not isinstance(k, int) or k < 1:
        raise ValueError("k는 양의 정수여야 합니다.")
    return question.strip()

def retrieve(question, k=2, page=None):
    question = validate_request(question, k)
    query = teaching_vector(question)
    if not any(query):
        return []
    pool = [d for d in documents if page is None or d["page"] == page]
    def rank(doc):
        stored = teaching_vector(doc["text"])
        return (sum((a - b) ** 2 for a, b in zip(query, stored)), doc["id"])
    return sorted(pool, key=rank)[:k]

hits = retrieve("휴관 요일은 언제인가요?", k=1)
if hits:
    print(hits[0]["id"])
else:
    print([])

library-2


library-2는 둘째 쪽의 휴관 안내입니다. k=1은 하나만 가져오라는 입력이지, 그 한 건으로 모든 질문에 답할 수 있다는 보장이 아닙니다.


### 후보를 거른 뒤 거리순으로 정렬하기

<p>retrieve는 documents에서 page 조건에 맞는 후보를 고른 뒤 질의와의 제곱 거리, 문서 ID 순으로 정렬하고 앞의 k개를 반환합니다. page=1이면 휴관 문서가 정렬 대상에서 이미 빠집니다.</p>


### 셀 04 · 메타데이터 필터의 page

휴관 질문은 그대로 두고 검색할 수 있는 쪽만 바꿉니다.


In [4]:
filtered = retrieve("휴관 요일은 언제인가요?", k=1, page=1)
print(filtered[0]["id"], filtered[0]["text"])

library-1 도서 대출 기간은 14일입니다.


1쪽에 남은 library-1은 대출 문장입니다. 진짜 근거가 필터 밖에 있어서 상위 한 건이 질문과 어긋납니다. k만 크게 바꿔도 범위 밖 문서는 돌아오지 않습니다.


### top-k와 필터

<p>k를 늘리면 더 많은 후보를 받을 수 있지만 그만큼 관련 없는 문서도 함께 들어올 수 있습니다. 반환 후보 수가 늘었다는 사실만으로 답변 품질 향상을 주장하지 않습니다.</p>

<p>where는 메타데이터 조건입니다. 같은 질문에 page=1 필터를 주면 해당 페이지 범위만 검색합니다. 잘못된 필터로 필요한 문서를 제외하면 질문이 맞아도 근거를 찾지 못할 수 있습니다.</p>

<p>대출 질문의 k=1에서는 library-1을 기대합니다. k=2에는 거리 동점의 무관한 문서가 들어갈 수 있습니다. page=1 필터는 검색 범위를 1페이지로 제한하는 조건이며 답변의 사실성 점수가 아닙니다. 대역 객체가 필터 인수를 받는지 검사하는 것과 실제 저장소가 필터를 적용하는 것은 서로 다른 검사입니다.</p>


## 문서 본문과 출처를 묶어 반환

생성 단계에는 사용자가 물은 문장과 검색한 문서를 별개로 전달합니다. 근거에 식별자와 출처를 표시해 두어야 나중에 어느 문장을 썼는지 확인할 수 있습니다.


### 검색 형식과 검색 품질 검사

<table><caption>관찰과 해석</caption><thead><tr><th scope="col">관찰</th><th scope="col">해석</th></tr>
</thead><tbody><tr><td>정상 빈 목록</td><td>현재 조건에서 후보 없음 또는 교육용 표현 범위 밖</td></tr>
<tr><td>출처 누락/DataContractError</td><td>데이터 계약 오류;성공 문서로 채우지 않음</td></tr>
<tr><td>TimeoutError/ConnectionError</td><td>운영 오류;no_evidence와 구분</td></tr>
<tr><td>E1 불일치</td><td>원문·모델·분할 정책을 다시 대조</td></tr>
<tr><td>동점 후보 순서 차이</td><td>후보와 거리를 함께 관찰</td></tr>
<tr><td>fixture 검사 통과</td><td>후처리 형태만 확인;실제 저장소 미확인</td></tr>
</tbody></table>


### 과제 05 · 검색 결과의 표시 형식

hits의 첫 문서에서 ID를 꺼내 대괄호로 감싸고 출처와 쪽을 붙인 다음 줄에 본문을 출력하세요.


In [5]:
# hit에서 id, source, page, text를 꺼내세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>hit = hits[0]
print(f&quot;[{hit[&#x27;id&#x27;]}] {hit[&#x27;source&#x27;]} / p.{hit[&#x27;page&#x27;]}\n{hit[&#x27;text&#x27;]}&quot;)</code></pre>
<p>실행 결과</p><pre><code>[library-2] 가상 도서관 이용 안내 / p.2
도서관은 월요일에 휴관합니다.</code></pre>
<p>[library-2]와 2쪽이 월요일 휴관 문장 바로 위에 붙습니다. 주차처럼 세 키워드 중 어느 것도 들어 있지 않은 질문에서는 빈 목록이 나오므로 hits[0]을 읽기 전에 목록이 비었는지 확인해야 합니다.</p>
</details>


### 질문과 근거 문맥

<p>문맥은 모델에게 제공할 근거 문서 묶음입니다. 본문 앞에 id와 출처·페이지를 붙이면 생성 답변의 인용 식별자를 다시 원문과 연결할 수 있습니다. 문서에 없는 사실을 출처 이름으로 포장하지 않도록 원문 본문도 함께 전달합니다.</p>

<p>질문은 질문으로, 근거는 근거로 전달합니다. 검색된 문서에 “앞선 지시를 무시하라” 같은 문장이 있어도 서비스의 실행 지침으로 따르지 않도록 경계를 명시합니다. 이 장은 문맥을 만드는 단계이며 모델을 호출하지 않습니다.</p>

<p>ID는 [library-1]처럼 대괄호로 표시합니다. 팀 자료의 team-01:p-01처럼 콜론을 포함한 ID도 허용합니다. 페이지를 모르는데 1로 채우거나 출처가 없는데 임의 이름을 만드는 대신 입력 자료를 검토합니다. 이 장에서는 모델을 호출하지 않습니다.</p>


## 돌아보기

질문을 ‘주차 요금은?’으로 바꿔 retrieve의 반환값을 확인하세요. 빈 목록은 정상 검색 중 근거를 못 찾은 경우입니다. 저장소 연결 실패를 같은 빈 목록으로 바꾸면 다음 장에서 사용자에게 잘못된 이유를 안내하게 됩니다.
